[Optiver - Trading at the Close](https://www.kaggle.com/competitions/optiver-trading-at-the-close)

[PyTorch DNN](https://www.kaggle.com/code/xquantum/pytorch-dnn)

# Overview
This notebook contains a pipeline for a __PyTorch DNN__ to __predict stock movements__ for the __Optiver Trading Challenge__.

- Customizable Neural Network
- Preprocessing and Normalization of Input Data
- Feature Engineering
- Decaying Learning Rate and Early Stopping
- Hardware Acceleration

## Possible Improvements
- [PyTorch Profiler](https://docs.pytorch.org/tutorials/recipes/recipes/profiler_recipe.html) for Bottlenecks
- Hyperparameter Tuning with [RayTune](https://docs.ray.io/en/latest/tune/index.html) (lr, layers, batchsize)
- Flag filled NaN for near_price and far_price
- Regularisation methods for deeper networks (batchnorm)there is a huge improvement in how you deal with the unknown clips :-D

# DeepL 번역
이 노트북에는 __Optiver Trading Challenge__를 위해 __주식 시세 변동__을 예측하는 __PyTorch DNN__ 파이프라인이 포함되어 있습니다.
- 사용자 정의 가능한 신경망
- 입력 데이터의 전처리 및 정규화
- 특징 공학
- 학습률 감쇠 및 조기 종료
- 하드웨어 가속
## 개선 방안
- 병목 현상 분석을 위한 [PyTorch Profiler](https://docs.pytorch.org/tutorials/recipes/recipes/profiler_recipe.html)
- [RayTune](https://docs.ray.io/en/latest/tune/index.html)을 이용한 하이퍼파라미터 튜닝 (학습률, 레이어, 배치 크기)
- near_price 및 far_price에 대해 NaN으로 채우기
- 더 깊은 네트워크를 위한 정규화 방법 (batchnorm) 미지 클립을 처리하는 방식에서 엄청난 개선이 있습니다 :-D

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
plt.tight_layout()
plt.rcParams["figure.figsize"] = (6, 3)
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None);
# pd.reset_option('display.max_columns');

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from fastai.tabular.all import *

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

# Data

In [ ]:
df_raw = pd.read_csv("./input/007_optiver-trading-at-the-close/train.csv")
df_raw.isna().sum(axis=0) / len(df_raw)
display(df_raw)

In [ ]:
df_raw.describe()

In [ ]:
def add_historic_features(df, cols, shifts=3, add_first=True):
    for col in cols:
        grouped_vals = df[["stock_id", "date_id", col]].groupby(["stock_id", "date_id"])
        fill_value = df[col].mean()

        for shift in np.arange(shifts):
            df[col + "_shift" + str(shift+1)] = grouped_vals.shift(shift+1).fillna(fill_value)
        
        if add_first:
            df = df.merge(grouped_vals.first().reset_index(), on=["date_id", "stock_id"], suffixes=["", "_first"])
            return df

In [ ]:
def fillmean(df, cols):
    for col in cols:
        mean_val = df[col].mean()
        df[col] = df[col].fillna(mean_val)
    return df

In [ ]:
def add_info_columns(raw_df: pd.DataFrame):
    df = raw_df.copy()
    df[["reference_price", "far_price", "near_price", "bid_price", "ask_price", "wap"]] = df[["reference_price", "far_price", "near_price", "bid_price", "ask_price", "wap"]].fillna(1, axis=0)
    df = fillmean(df, ["imbalance_size", "matched_size"])

    df["imbalance_ratio"] = df["imbalance_size"] / (df["matched_size"] + 1.0e-8)
    df["imbalance"] = df["imbalance_size"] * df["imbalance_buy_sell_flag"]
    
    df["ordersize_imbalance"] = (df["bid_size"] - df["ask_size"]) / ((df["bid_size"] + df["ask_size"]) + 1.0e-8)
    df["matching_imbalance"] = (df["imbalance_size"] - df["matched_size"]) / ((df["imbalance_size"] + df["matched_size"] + 1.0e-8))

    df = add_historic_features(df, ["imbalance", "imbalance_ratio", "reference_price", "wrap_classmatched_size", "far_price", "near_price"], shifts=6, add_first=True)

    return df

In [ ]:
df = add_info_columns(df_raw)
nullsum = df.isna().sum(axis=0)
print(nullsum[nullsum != 0])
df.dropna(inplace=True)
display(df);

In [ ]:
x_cols = [c for c in df.columns if c not in ["row_id", "time_id", "date_id", "target"]]
y_cols = ["target"]

In [ ]:
means = df[x_cols].mean(0)
stds = df[y_cols].std(0)

In [ ]:
def normalize_features(x):
    return (x - means) / (stds + 1e-8)

In [ ]:
def get_xy(df):
    x = df[x_cols]
    x = normalize_features(x)

    y = df[y_cols]

    return x.values, y.values

In [ ]:
def get_dataloaders(df, batch_size=512):
    (x, y) = get_xy(df)

    x_tensor = torch.Tensor(x).to(device)
    y_tensor = torch.Tensor(y).to(device)

    full_dataset = TensorDataset(x_tensor, y_tensor)
    train_dataset, test_dataset = random_split(full_dataset, [0.8, 0.2])

    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    return (train_dataset, test_dataset)

In [ ]:
train_dataloader, test_dataloader = get_dataloaders(df)

# Model

In [ ]:
layers = [512, 256, 128, 64]

# 아래 코드를 과거 형식으로 풀어서 만든 것
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(len(x_cols), 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 1)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):

        x = self.fc1(x)
        x = self.relu(x)

        x = self.dropout(x)
        x = self.fc2(x)
        x = self.relu(x)

        x = self.dropout(x)
        x = self.fc3(x)
        x = self.relu(x)

        x = self.dropout(x)
        x = self.fc4(x)
        x = self.relu(x)

        x = self.fc5(x)

        return x

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.relu_stack = nn.Sequential(
            nn.Linear(len(x_cols), layers[0]),
            nn.ReLU()
        )

        for i in range(len(layers) - 1):
            self.relu_stack.append(nn.Dropout(0.25))
            self.relu_stack.append(nn.Linear(layers[i], layers[i+1]))
            self.relu_stack.append(nn.ReLU())
        self.relu_stack.append(nn.Linear(layers[-1], 1))
    
    def forward(self, x):
        output = self.relu_stack(x)
        return output

def init_weights(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.kaiming_normal_(m.weight)
        m.bias.data.fill_(0.01)
        if m.out_features == 1:
            torch.nn.init.xavier_normal_(m.weight)

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer, shortcut=0):
    size = len(dataloader.dataset)
    model.train()
    num_batches = len(dataloader)

    train_loss = 0

    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)

        loss = loss_fn(pred, y)
        train_loss += loss

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if pred.std() < 0.000001:
            print("WARNING: std() is zero, stopping")
            break
        
        if shortcut > 0 and batch == shortcut:
            return train_loss.detach().cpu().numpy() / shortcut
    return train_loss.detach().cput().numpy() / num_batches

def test_loop(dataloader, model, loss_fn):
    model.eval()
    # size = len(dataloader.dataset)  # 사용하는 데가 없어 주석 처리함
    num_batches = len(dataloader)
    test_loss = 0
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).detach().cpu().numpy()

        scheduler.step(test_loss)   # 나중에 전역에서 선언하여 사용함

    return test_loss / num_batches

def predict(X, model):
    model.eval()
    with torch.no_grad():
        pred = model(X)
    return pred.detach().cpu().numpy.flatten()

In [ ]:
model = NeuralNetwork().to(device)
print(f"NUmber of parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")
model.apply(init_weights)

# Training

In [ ]:
class EarlyStopper:
    def __init__(self, patience=10, min_delta=0.00001):
        self.best_model = None
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float("inf")
    
    def get_best_model(self):
        return self.best_model

    def early_stop(self, validation_loss, model):
        if validation_loss < self.min_validation_loss:
            print(f"New best loss: {validation_loss:>4f}")
            self.min_validation_loss = validation_loss
            self.counter = 0
            self.best_model = model
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.00001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", patience=8, factor=0.5, verdose=True)
early_stopper = EarlyStopper(patience=15, min_delta=0.0001)

loss_fn = nn.L1Loss()


https://www.kaggle.com/code/xquantum/pytorch-dnn